### Teste de Hipótese

Após a realização da análise exploratória dos dados, esta etapa tem como objetivo verificar, por meio de testes estatísticos, se produtos que participaram de ações promocionais do tipo *display* apresentam quantidade média de unidades vendidas superior aos produtos que não participaram dessas ações.

In [30]:
# Imports e carregamento do dataset

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from scipy.stats import ttest_ind

import statsmodels.api as sm
from statsmodels.stats.weightstats import CompareMeans, DescrStatsW

df = pd.read_csv("../data/df_analytical.csv")

In [31]:
grupo_sem_promocao = df[df["HAS_DISPLAY"] == 0]["TOTAL_UNITS"]

grupo_com_promocao = df[df["HAS_DISPLAY"] == 1]["TOTAL_UNITS"]

Após a verificação dos pressupostos, será aplicado o **teste t de Welch** para comparar a média da quantidade total de unidades vendidas entre produtos que participaram de ações promocionais do tipo *display* e aqueles que não participaram.

Foi aplicado o teste t de Welch, pois é uma variação do teste t de Student utilizada quando a suposição de igualdade das variâncias não é atendida. E ele é mais adeuqado pois:
* os grupos são independentes;
* a variável resposta é quantitativa (TOTAL_UNITS);
* as variâncias são diferentes (Levene);
* as amostras são muito grandes.

**Aplicação do teste**

In [32]:
t_stat, p_valor = ttest_ind(
    grupo_com_promocao,
    grupo_sem_promocao,
    equal_var=False,
    alternative="greater"   # hipótese unilateral
)

print(f"Estatística t = {t_stat}")
print(f"p-valor = {p_valor}")

Estatística t = 10.248154873140892
p-valor = 7.0587982848187455e-25


**Interpretação automática**

In [33]:
alpha = 0.05

print(f"Nível de significância (α): {alpha}")

if p_valor < alpha:
    print("\nResultado:")
    print("Rejeita-se H₀.")
    print("Há evidências estatísticas de que produtos com ações de display apresentam maior média de unidades vendidas.")
else:
    print("\nResultado:")
    print("Não se rejeita H₀.")
    print("Não há evidências suficientes para afirmar que produtos com ações de display apresentam maior média de unidades vendidas.")

Nível de significância (α): 0.05

Resultado:
Rejeita-se H₀.
Há evidências estatísticas de que produtos com ações de display apresentam maior média de unidades vendidas.


Para tornar a análise mais robusta, também foram avaliados outros níveis de significância (α = 0,05; 0,01 e 0,001), equivalentes a níveis de confiança de 95%, 99% e 99,9%, respectivamente. Dessa forma, é possível verificar se a conclusão obtida permanece consistente mesmo sob critérios estatísticos mais rigorosos.

In [34]:
# Diferentes níveis de significância
alphas = [0.05, 0.01, 0.001]

for alpha in alphas:
    confianca = (1 - alpha) * 100

    if p_valor < alpha:
        print(f"Para o nível de significância α = {alpha}, rejeita-se H₀.")
        
    else:
        print(f"Para o nível de significância α = {alpha}, não se rejeita H₀.")

Para o nível de significância α = 0.05, rejeita-se H₀.
Para o nível de significância α = 0.01, rejeita-se H₀.
Para o nível de significância α = 0.001, rejeita-se H₀.


**Intervalo de Confiança da Diferença entre Médias**

Além do valor-p, será calculado o intervalo de confiança de 95% para a diferença entre as médias, permitindo estimar a magnitude da diferença observada entre os grupos.

In [35]:
grupo1 = DescrStatsW(grupo_com_promocao)
grupo2 = DescrStatsW(grupo_sem_promocao)

cm = CompareMeans(grupo1, grupo2)

ic = cm.tconfint_diff(
    alpha=0.05,
    usevar="unequal"
)

print(f"IC 95% = ({ic[0]:.2f}, {ic[1]:.2f})")

IC 95% = (24.25, 35.71)


**Tamanho do Efeito**

Embora o valor-p indique se existe diferença estatisticamente significativa entre os grupos, ele não informa a magnitude dessa diferença. Para isso, será calculado o coeficiente de **Cohen's d**, que mede o tamanho do efeito observado.

In [36]:
n1 = len(grupo_com_promocao)
n2 = len(grupo_sem_promocao)

s1 = grupo_com_promocao.std(ddof=1)
s2 = grupo_sem_promocao.std(ddof=1)

media1 = grupo_com_promocao.mean()
media2 = grupo_sem_promocao.mean()

desvio_pool = np.sqrt(
    ((n1-1)*s1**2 + (n2-1)*s2**2) /
    (n1+n2-2)
)

cohen_d = (media1-media2)/desvio_pool

print(f"Cohen's d = {cohen_d:.4f}")

Cohen's d = 0.1194


In [37]:
# Interpretação do tamanho do efeito

if abs(cohen_d) < 0.20:
    interpretacao = "efeito desprezível"
elif abs(cohen_d) < 0.50:
    interpretacao = "efeito pequeno"
elif abs(cohen_d) < 0.80:
    interpretacao = "efeito médio"
else:
    interpretacao = "efeito grande"

print(f"Interpretação: {interpretacao}")

Interpretação: efeito desprezível


### Resumo dos resultados

In [38]:
resultado = pd.DataFrame({
    "Métrica": [
        "Média (Com Display)",
        "Média (Sem Display)",
        "Estatística t",
        "p-valor",
        "Cohen's d"
    ],
    "Valor": [
        media1,
        media2,
        t_stat,
        p_valor,
        cohen_d
    ]
})

resultado

,Métrica,Valor
0,Média (Com Display),5.212424e+01
1,Média (Sem Display),2.214486e+01
2,Estatística t,1.024815e+01
3,p-valor,7.058798e-25
4,Cohen's d,1.193551e-01


### Considerações sobre o teste de hipótese

O teste t de Welch resultou em uma estatística **t = 10,25** e um **p-valor = 7,06 × 10⁻²⁵**, valor muito inferior aos níveis de significância adotados (0,05; 0,01 e 0,001). Dessa forma, rejeita-se a hipótese nula, indicando que existem evidências estatísticas de que produtos que participaram de ações promocionais do tipo *display* apresentam maior média de unidades vendidas em comparação aos produtos sem *display*.

O intervalo de confiança de 95% para a diferença entre as médias foi de **24,25 a 35,71 unidades**, indicando que, em média, produtos com *display* vendem entre aproximadamente 24 e 36 unidades a mais do que aqueles sem essa ação promocional.

Por outro lado, o tamanho do efeito medido pelo coeficiente de **Cohen's d = 0,119** foi classificado como **desprezível**, indicando que, embora a diferença entre os grupos seja estatisticamente significativa, sua magnitude prática é pequena. Esse resultado pode ser explicado pelo elevado tamanho da amostra, que aumenta a sensibilidade do teste estatístico para detectar diferenças mesmo quando elas possuem baixo impacto prático.

### Discussão

Embora os resultados indiquem que produtos participantes de ações promocionais do tipo *display* apresentam, em média, maior quantidade de unidades vendidas, o baixo valor do coeficiente de Cohen's *d* demonstra que essa diferença possui pequena magnitude prática. Isso sugere que a simples participação em uma ação de *display* não parece ser suficiente, por si só, para explicar grandes diferenças nas vendas dos produtos. Dessa forma, é provável que outros fatores, como características dos produtos, categoria, marca, preço, sazonalidade e intensidade das ações promocionais, também exerçam influência significativa sobre o desempenho das vendas, indicando que o comportamento observado resulta da combinação de múltiplos fatores e não exclusivamente da realização de ações de *display*.
